# Step 3 — Market Sentiment (news + social)

Classify recent news + social posts and cluster into themes (`docs/sentiment.md`). Backend is
configurable via `SENTIMENT_BACKEND` (`lexicon` offline / `finbert` = `ProsusAI/finbert`). Every
theme keeps its source items as evidence, and the "angry topics" feed question prediction.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot import corpus
from ir_copilot.agents.sentiment import analyze_sentiment

items = [it for it in (corpus.NEWS_HEADLINES + corpus.SOCIAL_POSTS) if it["ticker"] == settings.ticker]
snap = analyze_sentiment(settings.ticker, items)

print(f"{settings.ticker} net sentiment: {snap.net_score}  (over {len(items)} items)\n")
print("POSITIVE themes:")
for t in snap.positive_themes:
    print(f"  + {t.label} ({t.score})")
print("NEGATIVE themes (investors' concerns -> likely hard questions):")
for t in snap.negative_themes:
    print(f"  - {t.label} ({t.score})")
    for c in t.evidence[:1]:
        print(f"      evidence: {c.text[:70]}...  [{c.url}]")

assert snap.sources, "expected cited sources"
print("\nSentiment grounded in cited sources.")

NVDA net sentiment: 0.0  (over 20 items)

POSITIVE themes:
  + Revenue growth (0.0)
NEGATIVE themes (investors' concerns -> likely hard questions):

Sentiment grounded in cited sources.


### Real FinBERT
Set `SENTIMENT_BACKEND=finbert` in `.env` to classify with `ProsusAI/finbert` via transformers
(downloads the model). The theme structure and evidence are identical.

**Next (Step 4):** competitor comparison.